In [7]:
import tensorflow as tf
from tensorflow.keras.applications import DenseNet121
from tensorflow.keras.layers import Dense, GlobalAveragePooling2D, Reshape, Bidirectional, LSTM
from tensorflow.keras.models import Model
from tensorflow.keras.preprocessing.image import ImageDataGenerator
from tensorflow.keras.optimizers import Adam

from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score
import numpy as np
import random


In [8]:
DATASET_PATH = r"C:\Users\abhil\Desktop\model_z\Data"

IMG_SIZE = (224, 224)
BATCH_SIZE = 16
EPOCHS = 20

train_dir = DATASET_PATH + r"\Train"
test_dir  = DATASET_PATH + r"\Test"

datagen = ImageDataGenerator(rescale=1./255)

train_gen = datagen.flow_from_directory(
    train_dir,
    target_size=IMG_SIZE,
    batch_size=BATCH_SIZE,
    class_mode="categorical",
    shuffle=True
)

test_gen = datagen.flow_from_directory(
    test_dir,
    target_size=IMG_SIZE,
    batch_size=BATCH_SIZE,
    class_mode="categorical",
    shuffle=False
)


Found 5435 images belonging to 2 classes.
Found 1359 images belonging to 2 classes.


In [9]:
def build_model(learning_rate):
    base_model = DenseNet121(
        weights="imagenet",
        include_top=False,
        input_shape=(224, 224, 3)
    )

    for layer in base_model.layers:
        layer.trainable = False

    x = base_model.output
    x = GlobalAveragePooling2D()(x)
    x = Reshape((1, x.shape[-1]))(x)

    x = Bidirectional(LSTM(
        units=128,
        return_sequences=False,
        dropout=0.3,
        recurrent_dropout=0.3
    ))(x)

    output = Dense(train_gen.num_classes, activation="softmax")(x)

    model = Model(inputs=base_model.input, outputs=output)

    model.compile(
        optimizer=Adam(learning_rate=learning_rate),
        loss="categorical_crossentropy",
        metrics=["accuracy"]
    )

    return model


In [10]:
POPULATION_SIZE = 5
ITERATIONS = 5

LR_MIN = 1e-5
LR_MAX = 1e-3


In [11]:
particles = [random.uniform(LR_MIN, LR_MAX) for _ in range(POPULATION_SIZE)]
best_global_lr = particles[0]
best_global_score = 0


In [12]:
for iteration in range(ITERATIONS):
    print(f"\n⚛️ QPSO Iteration {iteration+1}/{ITERATIONS}")

    for i, lr in enumerate(particles):
        print(f" Testing LR: {lr:.6f}")

        model = build_model(lr)

        history = model.fit(
            train_gen,
            steps_per_epoch=50,
            epochs=1,
            validation_data=test_gen,
            validation_steps=20,
            verbose=0
        )


        score = max(history.history["val_accuracy"])

        if score > best_global_score:
            best_global_score = score
            best_global_lr = lr

    # Quantum position update (simplified)
    particles = [
        best_global_lr + random.uniform(-0.2, 0.2) * best_global_lr
        for _ in range(POPULATION_SIZE)
    ]

print("\n✅ Optimized Learning Rate:", best_global_lr)



⚛️ QPSO Iteration 1/5
 Testing LR: 0.000862
 Testing LR: 0.000318
 Testing LR: 0.000272
 Testing LR: 0.000684
 Testing LR: 0.000569

⚛️ QPSO Iteration 2/5
 Testing LR: 0.000614
 Testing LR: 0.000542
 Testing LR: 0.000553
 Testing LR: 0.000613
 Testing LR: 0.000620

⚛️ QPSO Iteration 3/5
 Testing LR: 0.000625
 Testing LR: 0.000443
 Testing LR: 0.000587
 Testing LR: 0.000533
 Testing LR: 0.000628

⚛️ QPSO Iteration 4/5
 Testing LR: 0.000355
 Testing LR: 0.000396
 Testing LR: 0.000442
 Testing LR: 0.000376
 Testing LR: 0.000526

⚛️ QPSO Iteration 5/5
 Testing LR: 0.000420
 Testing LR: 0.000399
 Testing LR: 0.000446
 Testing LR: 0.000406
 Testing LR: 0.000414

✅ Optimized Learning Rate: 0.00044269914886562985


In [13]:
final_model = build_model(best_global_lr)

history = final_model.fit(
    train_gen,
    epochs=EPOCHS,
    validation_data=test_gen
)


Epoch 1/20
340/340 ━━━━━━━━━━━━━━━━━━━━ 236s 670ms/step - accuracy: 0.6955 - loss: 0.5811 - val_accuracy: 0.8617 - val_loss: 0.3394
Epoch 2/20
340/340 ━━━━━━━━━━━━━━━━━━━━ 225s 663ms/step - accuracy: 0.8469 - loss: 0.3627 - val_accuracy: 0.8712 - val_loss: 0.2904
Epoch 3/20
340/340 ━━━━━━━━━━━━━━━━━━━━ 223s 657ms/step - accuracy: 0.8663 - loss: 0.3119 - val_accuracy: 0.9080 - val_loss: 0.2274
Epoch 4/20
340/340 ━━━━━━━━━━━━━━━━━━━━ 221s 650ms/step - accuracy: 0.8920 - loss: 0.2620 - val_accuracy: 0.9507 - val_loss: 0.1629
Epoch 5/20
340/340 ━━━━━━━━━━━━━━━━━━━━ 218s 642ms/step - accuracy: 0.9219 - loss: 0.2019 - val_accuracy: 0.9397 - val_loss: 0.1711
Epoch 6/20
340/340 ━━━━━━━━━━━━━━━━━━━━ 224s 658ms/step - accuracy: 0.9234 - loss: 0.1956 - val_accuracy: 0.9603 - val_loss: 0.1231
Epoch 7/20
340/340 ━━━━━━━━━━━━━━━━━━━━ 220s 648ms/step - accuracy: 0.9356 - loss: 0.1663 - val_accuracy: 0.9669 - val_loss: 0.1001
Epoch 8/20
340/340 ━━━━━━━━━━━━━━━━━━━━ 218s 642ms/step - accuracy: 0.9388 -

In [14]:
y_true = test_gen.classes
y_pred_prob = final_model.predict(test_gen)
y_pred = np.argmax(y_pred_prob, axis=1)

accuracy  = accuracy_score(y_true, y_pred)
precision = precision_score(y_true, y_pred, average="weighted")
recall    = recall_score(y_true, y_pred, average="weighted")
f1        = f1_score(y_true, y_pred, average="weighted")

print("DenseNet121 + BiLSTM + QIO Results:")
print("Accuracy :", accuracy)
print("Precision:", precision)
print("Recall   :", recall)
print("F1-score :", f1)


85/85 ━━━━━━━━━━━━━━━━━━━━ 47s 531ms/step
DenseNet121 + BiLSTM + QIO Results:
Accuracy : 0.9896983075791023
Precision: 0.9896983075791023
Recall   : 0.9896983075791023
F1-score : 0.9896983075791023
